# kael CUDA validation on Google Colab

This notebook runs `scripts/test_cuda.sh` on a real NVIDIA GPU and packs the report for download.

1. Select **Runtime > Change runtime type > T4 GPU** (any GPU works).
2. Run all cells. The validation builds kael three times with nvcc and takes 40 to 60 minutes on a T4. Each step prints its name as it starts.
3. Download `cuda-report.tgz` from the last cell.

The script builds kael with `KAEL_ENABLE_CUDA=ON`, runs the C++ device tests with and without cuBLAS, `compute-sanitizer`, the device benchmarks, MNIST on CPU and GPU, the character-level GPT, GPT-2 generation from C++ and Python, and the Python test suite.

In [ ]:
!nvidia-smi
!nvcc --version | tail -n 2
!cmake --version | head -n 1

## Get the code and the data

In [ ]:
!git clone --depth 1 https://github.com/JetBerri/kael.cpp
%cd kael.cpp
!git log --oneline -1

In [ ]:
!bash scripts/download_mnist.sh
!bash scripts/download_shakespeare.sh
!bash scripts/download_gpt2.sh

## Run the validation

In [ ]:
import os

jobs = os.cpu_count()
!pip install --quiet virtualenv
!CMAKE_BUILD_PARALLEL_LEVEL={jobs} bash scripts/test_cuda.sh native build-cuda cuda-report; echo "exit code: $?"

## Download the report

Exit code 0 means every step ran and passed, 1 means a step failed, and 3 means no step failed but some did not run. The archive holds `summary.txt` and one log per step.

In [ ]:
!tar czf cuda-report.tgz cuda-report
!cat cuda-report/summary.txt

from google.colab import files

files.download("cuda-report.tgz")